# Migration to the U.S.: combined country-year dataset

Builds one table with **one row per origin country per year** from the five files you downloaded, plus data the notebook fetches itself.

| Category | Source | Years |
|---|---|---|
| Green cards | DHS Yearbook 2023, Table 3 (by country of birth) | 2014–2023 |
| Refugees | DHS Yearbook 2024, Table 14 (by nationality) | 2015–2024 |
| Asylum grants | DHS Yearbook 2023, Table 17 (affirmative, by nationality) | 2014–2023 |
| Disasters | EM-DAT (natural disasters only) | 2000–2026 |
| Diaspora | Census ACS 2023, table B05006 | 2023 |
| Economy | World Bank (GDP per capita, unemployment, population) | fetched automatically |
| Violence, conflict, democracy | UNODC, UCDP, V-Dem via Our World in Data | fetched automatically |

**To run:** Runtime → Run all. When the upload button appears, select **all five files at once** (hold Ctrl/Cmd). Keep the original file names; the notebook uses them to tell the files apart.

## 1. Setup

In [ ]:
!pip install -q country_converter linearmodels

import io, re, logging, warnings
import numpy as np
import pandas as pd
import requests
import country_converter as coco
from google.colab import files

warnings.filterwarnings("ignore")
logging.getLogger("country_converter").setLevel(logging.ERROR)
cc = coco.CountryConverter()

# Your DHS data covers 2014-2024. Causes start a year earlier because they are lagged.
START_YEAR = 2013
END_YEAR   = 2024
LAG_YEARS  = 1   # conditions in year t are compared with migration in year t + 1

## 2. Readers for each file
Each function reads one of your downloads and returns tidy `iso3, year, value` rows. Country names are converted to standard 3-letter codes (`MEX`, `IND`, ...) so the tables line up across sources.

What they handle:
- **DHS tables:** skip the title rows and the regional subtotals, keep only countries, strip footnote numbers ("Unknown2"), and treat `X` (not applicable) as missing. DHS rounds counts to the nearest 10.
- **EM-DAT:** keep natural disasters and add them up per country per year.
- **Census ACS:** keep individual countries, skip subtotals like "Eastern Asia:", and read the year from the file name.

In [ ]:
# Names the converter gets wrong or can't place
NAME_FIXES = {
    "Korea, South": "KOR", "Korea, North": "PRK", "Korea": "KOR",
    "China, People's Republic": "CHN", "China, excluding Hong Kong and Taiwan": "CHN",
    "Burma": "MMR", "Burma (Myanmar)": "MMR",
    "Congo, Democratic Republic": "COD", "Democratic Republic of Congo (Zaire)": "COD",
    "Congo, Republic": "COG", "Congo": "COG",
    "Ivory Coast": "CIV", "Cote d'Ivoire": "CIV",
    "Kosovo": "XKX", "Micronesia, Federated States": "FSM", "Micronesia": "FSM",
}

def to_iso3(names):
    names = pd.Series(names).astype(str).str.strip()
    fixed = names.map(NAME_FIXES)
    rest = names[fixed.isna()]
    if len(rest):
        fixed.loc[rest.index] = cc.pandas_convert(series=rest, to="ISO3", not_found=np.nan)
    return fixed.values

def read_dhs_table(path, sheet, name):
    """DHS Yearbook country tables: headers on Excel row 6, countries listed below the 'COUNTRY' row."""
    df = pd.read_excel(path, sheet_name=sheet, header=5)
    first = df.columns[0]
    labels = df[first].astype(str).str.strip()
    start = labels.str.upper().eq("COUNTRY").idxmax() + 1          # skip the REGION block
    df = df.loc[start:].copy()
    df[first] = df[first].astype(str).str.strip().str.replace(r"\d+$", "", regex=True)  # drop footnote numbers
    df = df[~df[first].str.contains(r"^Total$|All other|Unknown|former|^nan$|^Notes|^Source|^X |^\d", regex=True)]
    year_cols = [c for c in df.columns[1:] if re.fullmatch(r"(19|20)\d\d(\.0)?", str(c))]
    long = df.melt(id_vars=[first], value_vars=year_cols, var_name="year", value_name=name)
    long["year"] = long["year"].astype(float).astype(int)
    long[name] = pd.to_numeric(long[name].replace({"X": np.nan, "D": np.nan, "-": 0}), errors="coerce")
    long["iso3"] = to_iso3(long[first])
    bad = sorted(long.loc[long["iso3"].isna(), first].unique())
    if bad:
        print(f"  {name}: could not match {len(bad)} names (dropped): {bad}")
    long = long.dropna(subset=["iso3"]).groupby(["iso3", "year"], as_index=False)[name].sum(min_count=1)
    print(f"Loaded {name}: {long['iso3'].nunique()} countries, {long['year'].min()}-{long['year'].max()}")
    return long

def read_emdat(path):
    em = pd.read_excel(path)
    em = em[em["Disaster Group"] == "Natural"]
    out = (em.groupby(["ISO", "Start Year"])
             .agg(disaster_deaths=("Total Deaths", "sum"),
                  disaster_affected=("Total Affected", "sum"),
                  disaster_count=("DisNo.", "count"))
             .reset_index().rename(columns={"ISO": "iso3", "Start Year": "year"}))
    print(f"Loaded disasters: {out['iso3'].nunique()} countries, {out['year'].min()}-{out['year'].max()}")
    return out

def read_acs_b05006(path):
    """Census ACS table B05006 (foreign-born by country of birth). Year comes from the file name."""
    m = re.search(r"ACS\w*?(\d{4})", str(path))
    year = int(m.group(1)) if m else None
    raw = pd.read_csv(path)
    lab = raw.iloc[:, 0].astype(str)
    est = pd.to_numeric(raw.iloc[:, 1].astype(str).str.replace(",", ""), errors="coerce")
    name = lab.str.strip()
    keep = ~name.str.endswith(":")                                         # ':' marks a subtotal
    keep &= ~name.str.contains(r"^Other |n\.e\.c\.|West Indies|Azores|England|Scotland|excluding England", regex=True)
    df = pd.DataFrame({"name": name[keep], "diaspora_in_us": est[keep]})
    # Subtotals that are really single countries
    for label, iso in [("United Kingdom (inc. Crown Dependencies):", "GBR")]:
        row = name.eq(label)
        if row.any():
            df = pd.concat([df, pd.DataFrame({"name": [label], "diaspora_in_us": est[row].values})])
    df["iso3"] = to_iso3(df["name"].str.rstrip(":").str.replace(r" \(inc\..*\)", "", regex=True))
    bad = sorted(df.loc[df["iso3"].isna(), "name"])
    if bad:
        print(f"  diaspora: could not match {bad}")
    df = df.dropna(subset=["iso3"]).groupby("iso3", as_index=False)["diaspora_in_us"].sum()
    df["year"] = year
    print(f"Loaded diaspora ({year}): {df['iso3'].nunique()} countries")
    return df

## 3. Upload your five files

In [ ]:
uploaded = files.upload()

def find(keyword):
    match = [n for n in uploaded if keyword.lower() in n.lower()]
    return match[0] if match else None

paths = {
    "green_cards": find("lawful_permanent_residents"),
    "refugees":    find("refugees"),
    "asylum":      find("asylees"),
    "disasters":   find("emdat"),
    "diaspora":    [n for n in uploaded if "b05006" in n.lower()],
}
for k, v in paths.items():
    print(f"{k:12s} -> {v if v else 'NOT FOUND (this column will be skipped)'}")

In [ ]:
green_cards = read_dhs_table(paths["green_cards"], "Table 3",  "green_cards")   if paths["green_cards"] else None
refugees    = read_dhs_table(paths["refugees"],    "Table 14", "refugees")      if paths["refugees"]    else None
asylum      = read_dhs_table(paths["asylum"],      "Table 17", "asylum_grants") if paths["asylum"]      else None
disasters   = read_emdat(paths["disasters"])                                     if paths["disasters"]   else None
diaspora    = pd.concat([read_acs_b05006(p) for p in paths["diaspora"]]) if paths["diaspora"] else None

## 4. Automatic downloads
If any of these fail, the notebook prints a message and carries on without that column.

In [ ]:
VALID_ISO3 = set(cc.data["ISO3"])

def world_bank(code, name):
    url = (f"https://api.worldbank.org/v2/country/all/indicator/{code}"
           f"?format=json&per_page=20000&date={START_YEAR}:{END_YEAR}")
    data = requests.get(url, timeout=60).json()[1]
    return pd.DataFrame([{"iso3": d["countryiso3code"], "year": int(d["date"]), name: d["value"]}
                         for d in data if d["countryiso3code"] in VALID_ISO3])

wb = None
for code_, name in {"NY.GDP.PCAP.KD": "gdp_per_capita",
                    "SL.UEM.TOTL.ZS": "unemployment_pct",
                    "SP.POP.TOTL":    "population"}.items():
    try:
        df = world_bank(code_, name)
        wb = df if wb is None else wb.merge(df, on=["iso3", "year"], how="outer")
        print(f"Loaded {name}")
    except Exception as e:
        print(f"World Bank {name} failed: {type(e).__name__}")

def owid(slugs, name):
    for slug in slugs:
        try:
            url = f"https://ourworldindata.org/grapher/{slug}.csv?v=1&csvType=full&useColumnShortNames=true"
            df = pd.read_csv(url, storage_options={"User-Agent": "Mozilla/5.0"})
            value_col = [c for c in df.columns if c not in ("Entity", "Code", "Year")][0]
            df = df.rename(columns={"Code": "iso3", "Year": "year", value_col: name})
            df = df[df["iso3"].isin(VALID_ISO3)][["iso3", "year", name]]
            print(f"Loaded {name} ({slug})")
            return df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)]
        except Exception as e:
            print(f"  {slug} failed: {type(e).__name__}")
    print(f"Skipping {name}.")
    return None

homicide  = owid(["homicide-rate-unodc", "homicide-rate"], "homicide_rate")
conflict  = owid(["deaths-in-intrastate-conflicts", "deaths-in-armed-conflicts-based-on-where-they-occurred"], "conflict_deaths")
democracy = owid(["electoral-democracy-index", "electoral-democracy"], "democracy_index")

## 5. Combine
- Migration columns stay in their own year.
- Causes are shifted forward by `LAG_YEARS` and end in `_lag1`, so the 2016 row holds 2016 migration beside 2015 conditions.
- **Diaspora:** your Census file covers one year (2023), so it's attached to every year as a fixed country trait (`diaspora_in_us_2023`). If you add more ACS years, it becomes a lagged yearly column instead.
- No EM-DAT or UCDP record means no recorded disaster or war deaths, so those gaps are filled with 0.

In [ ]:
outcomes = [o for o in (green_cards, refugees, asylum) if o is not None]
panel = outcomes[0]
for o in outcomes[1:]:
    panel = panel.merge(o, on=["iso3", "year"], how="outer")

def add_lagged(panel, df):
    if df is None:
        return panel
    df = df[(df["year"] >= START_YEAR) & (df["year"] <= END_YEAR)].copy()
    df["year"] += LAG_YEARS
    df = df.rename(columns={c: f"{c}_lag{LAG_YEARS}" for c in df.columns if c not in ("iso3", "year")})
    return panel.merge(df, on=["iso3", "year"], how="left")

for cause in (wb, homicide, conflict, democracy, disasters):
    panel = add_lagged(panel, cause)

if diaspora is not None:
    if diaspora["year"].nunique() == 1:
        yr = int(diaspora["year"].iloc[0])
        panel = panel.merge(diaspora[["iso3", "diaspora_in_us"]].rename(columns={"diaspora_in_us": f"diaspora_in_us_{yr}"}),
                            on="iso3", how="left")
    else:
        panel = add_lagged(panel, diaspora)

L = f"_lag{LAG_YEARS}"
for col in [f"conflict_deaths{L}", f"disaster_deaths{L}", f"disaster_affected{L}", f"disaster_count{L}"]:
    if col in panel:
        panel[col] = panel[col].fillna(0)
if f"population{L}" in panel:
    for o in ("green_cards", "refugees", "asylum_grants"):
        if o in panel:
            panel[f"{o}_per_100k"] = panel[o] / panel[f"population{L}"] * 1e5

panel.insert(1, "country", cc.pandas_convert(series=panel["iso3"], src="ISO3", to="name_short", not_found=None))
panel = panel.sort_values(["country", "year"]).reset_index(drop=True)
print(panel.shape)
panel.head(10)

## 6. Check coverage
Percent of rows with data in each column. Refugee and asylum columns are mostly empty because only some countries send refugees or asylum seekers in a given year; that's expected.

In [ ]:
print((panel.notna().mean() * 100).round(1).sort_values().to_string())

## 7. Save and download

In [ ]:
panel.to_csv("migration_panel.csv", index=False)
files.download("migration_panel.csv")

## 8. First look at the analysis
Two regressions on log green cards, with all predictors standardized so their sizes can be compared directly:

1. **Within-country (fixed effects):** does migration from a country rise when *its own* conditions change? Fixed country traits, like distance or diaspora size, are absorbed here.
2. **Across countries (year effects only):** which countries send more people overall? This is where diaspora size can be tested.

In [ ]:
from linearmodels.panel import PanelOLS

outcome = "green_cards"
df = panel.copy()
df["log_outcome"] = np.log1p(df[outcome])
for col in df.columns:
    if any(k in col for k in ("gdp_per_capita", "conflict_deaths", "disaster_deaths", "disaster_affected", "diaspora")):
        df[col] = np.log1p(df[col])

time_varying = [c for c in df.columns if c.endswith(L) and not c.startswith("population")]
fixed_traits = [c for c in df.columns if c.startswith("diaspora_in_us_") and not c.endswith(L)]

def standardized_fit(cols, entity_effects):
    d = df.dropna(subset=["log_outcome"] + cols).set_index(["iso3", "year"])
    X = (d[cols] - d[cols].mean()) / d[cols].std()
    res = PanelOLS(d["log_outcome"], X, entity_effects=entity_effects, time_effects=True,
                   drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)
    print(f"{len(d)} country-years, {d.index.get_level_values(0).nunique()} countries")
    return res.summary.tables[1]

print("=== 1. Within-country model ===")
print(standardized_fit(time_varying, entity_effects=True))
print("\n=== 2. Across-country model ===")
print(standardized_fit(time_varying + fixed_traits, entity_effects=False))